# O-Nakala Core Workshop: Complete Workflow Demonstration

**Welcome to the O-Nakala Core hands-on workshop!**

This notebook demonstrates the complete workflow for managing research data with the NAKALA repository using the O-Nakala Core Python library.

## 🎯 Learning Objectives

By the end of this workshop, you will know how to:
1. Install and configure O-Nakala Core
2. Upload research datasets with metadata
3. Create and manage thematic collections
4. Perform quality analysis and curation
5. Use both CLI tools and Python API

## 📚 Workshop Structure

- **Part 1**: Installation and Setup
- **Part 2**: Data Upload Workflow
- **Part 3**: Collection Management
- **Part 4**: Quality Analysis and Curation
- **Part 5**: Advanced Usage and Best Practices

---

**⚠️ Important**: We'll use the NAKALA test environment for safe experimentation. No real data will be published.

## Part 1: Installation and Setup

First, let's install O-Nakala Core from PyPI and set up our environment.

In [21]:
# Install O-Nakala Core from PyPI with CLI tools
import subprocess
import sys

# Install with proper bracket handling for CLI extras
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'o-nakala-core[cli]', '--quiet'], 
                       capture_output=True, text=True)

if result.returncode == 0:
    print("✅ O-Nakala Core installed successfully from PyPI!")
    print("📦 Package includes: upload, collection, curator, and user-info tools")
else:
    print(f"❌ Installation failed: {result.stderr}")
    print("Trying alternative installation method...")
    # Alternative method: install base package + CLI dependencies separately
    result2 = subprocess.run([sys.executable, '-m', 'pip', 'install', 'o-nakala-core', '--quiet'], 
                            capture_output=True, text=True)
    if result2.returncode == 0:
        # Install CLI dependencies manually
        result3 = subprocess.run([sys.executable, '-m', 'pip', 'install', 
                                'click>=8.0.0', 'rich>=13.0.0', 'python-dotenv>=1.0.0', '--quiet'],
                                capture_output=True, text=True)
        if result3.returncode == 0:
            print("✅ O-Nakala Core installed successfully with CLI dependencies!")
            print("📦 Package includes: upload, collection, curator, and user-info tools")
        else:
            print(f"❌ CLI dependencies failed: {result3.stderr}")
    else:
        print(f"❌ Alternative installation also failed: {result2.stderr}")
        print("💡 Try running: !pip install \"o-nakala-core[cli]\" manually")

✅ O-Nakala Core installed successfully from PyPI!
📦 Package includes: upload, collection, curator, and user-info tools


In [22]:
# Verify installation by checking available CLI commands
import subprocess
import sys

commands = ['o-nakala-upload', 'o-nakala-collection', 'o-nakala-curator', 'o-nakala-user-info']

print("🔍 Verifying CLI commands:")
for cmd in commands:
    try:
        result = subprocess.run([cmd, '--help'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            print(f"  ✅ {cmd} - Available")
        else:
            print(f"  ❌ {cmd} - Error")
    except Exception as e:
        print(f"  ❌ {cmd} - Failed: {e}")

print("\n🐍 Python imports:")
try:
    import o_nakala_core
    print("  ✅ o_nakala_core - Core library imported")
    
    from o_nakala_core.upload import NakalaUploadClient
    from o_nakala_core.collection import NakalaCollectionClient  
    from o_nakala_core.curator import NakalaCuratorClient
    print("  ✅ All main modules imported successfully")
    
except ImportError as e:
    print(f"  ❌ Import error: {e}")

🔍 Verifying CLI commands:
  ❌ o-nakala-upload - Failed: [Errno 2] No such file or directory: 'o-nakala-upload'
  ❌ o-nakala-collection - Failed: [Errno 2] No such file or directory: 'o-nakala-collection'
  ❌ o-nakala-curator - Failed: [Errno 2] No such file or directory: 'o-nakala-curator'
  ❌ o-nakala-user-info - Failed: [Errno 2] No such file or directory: 'o-nakala-user-info'

🐍 Python imports:
  ✅ o_nakala_core - Core library imported
  ❌ Import error: No module named 'o_nakala_core.upload'


In [23]:
# Set up environment for NAKALA test API
import os

# Try to import rich for better formatting, fallback to basic if not available
try:
    from rich.console import Console
    from rich.panel import Panel
    console = Console()
    rich_available = True
except ImportError:
    console = None
    rich_available = False
    print("⚠️  Rich not available - using basic formatting")

# Configure test environment (safe for workshops)
TEST_API_KEY = "33170cfe-f53c-550b-5fb6-4814ce981293"
TEST_BASE_URL = "https://apitest.nakala.fr"

os.environ['NAKALA_API_KEY'] = TEST_API_KEY
os.environ['NAKALA_BASE_URL'] = TEST_BASE_URL

if rich_available:
    console.print(Panel.fit(
        "[bold green]Environment Configuration[/bold green]\n\n"
        f"🔑 API Key: {TEST_API_KEY[:8]}...{TEST_API_KEY[-8:]}\n"
        f"🌐 Base URL: {TEST_BASE_URL}\n\n"
        "[yellow]⚠️  Test Environment[/yellow]: Safe for experimentation\n"
        "No real data will be permanently published",
        title="🚀 Setup Complete"
    ))
else:
    print("🚀 Setup Complete")
    print(f"🔑 API Key: {TEST_API_KEY[:8]}...{TEST_API_KEY[-8:]}")
    print(f"🌐 Base URL: {TEST_BASE_URL}")
    print("⚠️  Test Environment: Safe for experimentation")
    print("No real data will be permanently published")

print("\n✅ Environment configured for NAKALA test API")
print("🔒 Using validated test credentials - safe for workshop use")

╭────────────── 🚀 Setup Complete ──────────────╮
│ Environment Configuration                     │
│                                               │
│ 🔑 API Key: 33170cfe...ce981293               │
│ 🌐 Base URL: https://apitest.nakala.fr        │
│                                               │
│ ⚠️  Test Environment: Safe for experimentation │
│ No real data will be permanently published    │
╰───────────────────────────────────────────────╯


✅ Environment configured for NAKALA test API
🔒 Using validated test credentials - safe for workshop use


## Part 2: Data Upload Workflow

Now let's explore the data upload functionality using the example dataset from the workflow documentation.

In [24]:
# First, let's examine the sample dataset structure
import pandas as pd
from pathlib import Path

# Path to the sample dataset (relative to notebook location)
sample_dataset_path = Path("../sample_dataset")
workflow_docs_path = Path("../workflow_documentation")

console.print(Panel.fit(
    "[bold blue]Workshop Dataset Structure[/bold blue]\n\n"
    "📁 Sample Dataset: Multi-language research project\n"
    "📋 5 data categories: code, data, documents, images, presentations\n"
    "🌍 Multilingual metadata: French/English\n"
    "📊 14 files across different research outputs",
    title="📚 Dataset Overview"
))

# Display the dataset configuration
if (sample_dataset_path / "folder_data_items.csv").exists():
    df = pd.read_csv(sample_dataset_path / "folder_data_items.csv")
    print(f"\n📋 Dataset Configuration ({len(df)} items):")
    
    # Show available columns
    print(f"📊 Available columns: {list(df.columns)}")
    
    # Display key fields that exist
    available_cols = []
    for col in ['title', 'type', 'file']:
        if col in df.columns:
            available_cols.append(col)
    
    if available_cols:
        print(f"\n📋 Dataset Preview:")
        print(df[available_cols].head())
    
    # Analyze file categories if 'file' column exists
    if 'file' in df.columns:
        print(f"\n📊 File Categories:")
        # Extract directory from file paths
        file_dirs = df['file'].str.split('/').str[1]  # Get second part (after 'files/')
        category_counts = file_dirs.value_counts()
        for category, count in category_counts.items():
            print(f"  📁 {category}: {count} items")
else:
    print("⚠️  Sample dataset not found. Please ensure you're running from the correct directory.")

╭─────────────────────── 📚 Dataset Overview ────────────────────────╮
│ Workshop Dataset Structure                                         │
│                                                                    │
│ 📁 Sample Dataset: Multi-language research project                 │
│ 📋 5 data categories: code, data, documents, images, presentations │
│ 🌍 Multilingual metadata: French/English                           │
│ 📊 14 files across different research outputs                      │
╰────────────────────────────────────────────────────────────────────╯


📋 Dataset Configuration (5 items):
📊 Available columns: ['file', 'status', 'type', 'title', 'alternative', 'author', 'contributor', 'date', 'license', 'description', 'keywords', 'language', 'temporal', 'spatial', 'accessRights', 'identifier', 'rights']

📋 Dataset Preview:
                                               title  \
0                  fr:Fichiers de code|en:Code Files   
1           fr:Données de recherche|en:Research Data   
2    fr:Documents de recherche|en:Research Documents   
3         fr:Collection d'images|en:Image Collection   
4  fr:Matériaux de présentation|en:Presentation M...   

                                        type                  file  
0  http://purl.org/coar/resource_type/c_5ce6           files/code/  
1  http://purl.org/coar/resource_type/c_ddb1           files/data/  
2  http://purl.org/coar/resource_type/c_18cf      files/documents/  
3  http://purl.org/coar/resource_type/c_c513         files/images/  
4  http://purl.org/coar/resource_type/c_18cf

In [25]:
# Demonstrate the Python API for upload
from o_nakala_core.common.config import NakalaConfig
from o_nakala_core.upload import NakalaUploadClient
import tempfile
import json

# Initialize the configuration
config = NakalaConfig()
config.api_key = TEST_API_KEY
config.base_url = TEST_BASE_URL

console.print(Panel.fit(
    "[bold green]Upload Client Initialization[/bold green]\n\n"
    "🔧 Configuration: Test environment\n"
    "📡 Connection: Ready for API calls\n"
    "🛡️  Validation: Enabled for safety",
    title="⚙️ Client Setup"
))

# Create upload client
upload_client = NakalaUploadClient(config)

print("✅ Upload client initialized")
print("🔗 Connected to NAKALA test environment")
print("📋 Ready for data upload demonstrations")

ModuleNotFoundError: No module named 'o_nakala_core.common.config'

In [ ]:
# Demonstrate CLI usage for uploads (validate-only mode for safety)
console.print(Panel.fit(
    "[bold yellow]CLI Upload Demonstration[/bold yellow]\n\n"
    "This demonstrates the command-line interface\n"
    "Using --validate-only mode for safe demonstration",
    title="🖥️ Command Line Interface"
))

# Show the CLI command that would be used (corrected format)
cli_command = f"""
o-nakala-upload \\
  --api-key {TEST_API_KEY} \\
  --dataset ../sample_dataset \\
  --base-path ../sample_dataset \\
  --mode folder \\
  --folder-config ../sample_dataset/folder_data_items.csv \\
  --validate-only \\
  --output upload_results.csv
"""

print("📝 CLI Upload Command:")
print(cli_command)

print("\n🔍 Command Breakdown:")
print("  --dataset: Directory containing data files")
print("  --base-path: Root directory containing files to upload")
print("  --mode folder: Upload files from folder structure")
print("  --folder-config: CSV file with metadata configuration")
print("  --validate-only: Validate without actually uploading")
print("  --output: Save results to CSV file")

print("\n💡 Note: For folder mode, both --dataset and --folder-config are required")

## Part 3: Collection Management

Learn how to create and manage thematic collections to organize your uploaded datasets.

In [ ]:
# Explore collection configuration
from o_nakala_core.collection import NakalaCollectionClient

console.print(Panel.fit(
    "[bold purple]Collection Management[/bold purple]\n\n"
    "📚 Collections organize related datasets\n"
    "🏷️  Thematic grouping for research projects\n"
    "🔗 Link datasets by topic or methodology",
    title="📁 Collections Overview"
))

# Check if collection configuration exists
collections_file = sample_dataset_path / "folder_collections.csv"
if collections_file.exists():
    collections_df = pd.read_csv(collections_file)
    print(f"\n📋 Collection Configuration ({len(collections_df)} collections):")
    
    for idx, row in collections_df.iterrows():
        print(f"\n📁 Collection {idx + 1}: {row['title']}")
        print(f"   📝 Description: {row['description'][:100]}...")
        if 'keywords' in row and pd.notna(row['keywords']):
            print(f"   🏷️  Keywords: {row['keywords']}")
        
# Initialize collection client
collection_client = NakalaCollectionClient(config)
print("\n✅ Collection client initialized")
print("🔗 Ready for collection management")

In [ ]:
# Demonstrate collection creation CLI command
console.print(Panel.fit(
    "[bold cyan]Collection Creation Demo[/bold cyan]\n\n"
    "Creating collections from uploaded datasets\n"
    "Using configuration file for batch operations",
    title="🏗️ Collection Builder"
))

collection_cli_command = f"""
o-nakala-collection \\
  --api-key {TEST_API_KEY} \\
  --from-folder-collections ../sample_dataset/folder_collections.csv \\
  --from-upload-output upload_results.csv \\
  --validate-only
"""

print("📝 CLI Collection Command:")
print(collection_cli_command)

print("\n🔍 Command Breakdown:")
print("  --from-folder-collections: Use collection configuration file")
print("  --from-upload-output: Link to previously uploaded datasets")
print("  --validate-only: Validate collection setup without creating")

print("\n📊 Workflow Integration:")
print("  1. Upload datasets → get upload_results.csv")
print("  2. Create collections → link datasets by theme")
print("  3. Organize research outputs logically")

## Part 4: Quality Analysis and Curation

Explore the powerful curation tools for metadata quality analysis and enhancement.

In [ ]:
# Demonstrate quality analysis
from o_nakala_core.curator import NakalaCuratorClient

console.print(Panel.fit(
    "[bold red]Quality Analysis & Curation[/bold red]\n\n"
    "🔍 Metadata quality assessment\n"
    "📊 Completeness and consistency analysis\n"
    "🔧 Batch modification capabilities\n"
    "✅ Validation and enhancement tools",
    title="🎯 Curator Tools"
))

# Initialize curator client
curator_client = NakalaCuratorClient(config)

print("✅ Curator client initialized")
print("🔍 Ready for quality analysis")

# Show available curator functions
print("\n🛠️ Available Curator Functions:")
print("  📊 Quality reports - Analyze metadata completeness")
print("  🔧 Batch modifications - Update multiple records")
print("  ✅ Validation - Check metadata consistency")
print("  📈 Enhancement suggestions - Improve data quality")

In [ ]:
# Demonstrate quality report CLI command
console.print(Panel.fit(
    "[bold orange]Quality Report Generation[/bold orange]\n\n"
    "Comprehensive analysis of your NAKALA data\n"
    "Identifies areas for improvement",
    title="📋 Quality Assessment"
))

quality_cli_command = f"""
o-nakala-curator \\
  --api-key {TEST_API_KEY} \\
  --quality-report \\
  --scope collections \\
  --output quality_report.json
"""

print("📝 CLI Quality Report Command:")
print(quality_cli_command)

print("\n🔍 Report Features:")
print("  📊 Metadata completeness analysis")
print("  🏷️  Field usage statistics")
print("  🌍 Language distribution")
print("  ⚠️  Missing or inconsistent data identification")
print("  💡 Enhancement suggestions")

print("\n📈 Quality Metrics:")
print("  • Title completeness")
print("  • Description quality")
print("  • Keyword coverage")
print("  • Creator information")
print("  • Subject classification")

### 🔧 Hands-on: Batch Modification Demonstration

Now let's see batch modification in action\! We'll create a modification template and safely test changes to metadata.

In [ ]:
# Create a sample modification template for demonstration
import pandas as pd
import tempfile
import os
from pathlib import Path

console.print(Panel.fit(
    "[bold cyan]Batch Modification Demo[/bold cyan]\n\n"
    "🔧 Creating modification template\n"
    "📋 Safe validation with dry-run\n"
    "✅ Hands-on metadata modification",
    title="🛠️ Practical Demo"
))

# Create a sample modification template using correct curator format
# In a real scenario, you would get these IDs from your actual NAKALA data
modification_data = {
    'id': [
        'sample-id-1',  # These would be real NAKALA identifiers
        'sample-id-2',
        'sample-id-3'
    ],
    'action': [
        'modify',
        'modify',
        'modify'
    ],
    'new_title': [
        'Updated Research Dataset Title',
        None,  # No title change for this item
        None
    ],
    'new_description': [
        None,  # No description change for this item
        'Additional description for enhanced discoverability',
        None
    ],
    'new_keywords': [
        None,  # No keywords change for this item
        None,
        'workshop;demonstration;batch-modification'
    ]
}

modification_df = pd.DataFrame(modification_data)
print("📋 Sample Modification Template (Correct Format):")
print(modification_df)

# Save to temporary file for demonstration
temp_dir = tempfile.mkdtemp()
template_file = Path(temp_dir) / "workshop_modifications.csv"
modification_df.to_csv(template_file, index=False)

print(f"\n💾 Template saved to: {template_file}")
print("✅ Ready for batch modification testing")
print("\n📝 Template Format:")
print("  • id: NAKALA identifier")
print("  • action: 'modify' for updates")
print("  • new_[field]: New values for specific fields")

In [ ]:
# Demonstrate template creation and validation
console.print(Panel.fit(
    "[bold yellow]Template Structure[/bold yellow]\n\n"
    "📋 Required fields for batch modification:\n"
    "• id: NAKALA data identifier\n"
    "• action: 'modify' for updates\n"
    "• new_[field]: New values for specific metadata fields\n"
    "• current_[field]: Optional current values for validation",
    title="📝 Template Guide"
))

print("🔍 Template Breakdown:")
print("\n📊 Available actions:")
print("  • modify: Update metadata fields")
print("  • (other actions may be available - check documentation)")

print("\n🏷️ Common modification fields:")
print("  • new_title: Update dataset titles")
print("  • new_description: Enhance descriptions")
print("  • new_keywords: Add or update discovery terms")
print("  • new_author: Update creator information")
print("  • new_date: Modify temporal information")
print("  • new_license: Update license information")

print("\n💡 Template Tips:")
print("  • Use None/empty for fields you don't want to change")
print("  • Each row represents one dataset modification")
print("  • Multiple fields can be updated per dataset")
print("  • Multilingual format: 'fr:French|en:English'")

print("\n⚠️  Safety Note:")
print("  Always use --dry-run first to validate changes!")
print("  This prevents accidental data modifications.")

In [ ]:
# Demonstrate CLI dry-run validation
console.print(Panel.fit(
    "[bold orange]Dry-Run Validation[/bold orange]\n\n"
    "🔍 Validate modifications safely\n"
    "⚡ No actual changes made\n"
    "📊 Preview of what would happen",
    title="🛡️ Safe Testing"
))

# Show the CLI command for dry-run validation
dry_run_command = f"""
o-nakala-curator \\
  --api-key {TEST_API_KEY} \\
  --batch-modify {template_file} \\
  --dry-run \\
  --verbose
"""

print("📝 Dry-Run Command:")
print(dry_run_command)

print("🔍 What dry-run shows:")
print("  ✅ Template validation results")
print("  📋 List of records that would be modified")
print("  🔍 Preview of changes for each field")
print("  ⚠️  Any validation errors or warnings")
print("  📊 Summary of modification scope")

print("\n💡 Best Practice:")
print("  1. Create template with sample data")
print("  2. Test with --dry-run first")
print("  3. Review all proposed changes")
print("  4. Execute only after validation")
print("  5. Verify results after modification")

In [ ]:
# Demonstrate real batch modification (simulated for safety)
console.print(Panel.fit(
    "[bold green]Batch Modification Execution[/bold green]\n\n"
    "⚡ Real modifications (test environment)\n"
    "📊 Live metadata updates\n"
    "🔍 Results verification",
    title="🚀 Live Demo"
))

# For workshop safety, we'll simulate the execution
# In a real scenario, you would remove --dry-run
execution_command = f"""
o-nakala-curator \\
  --api-key {TEST_API_KEY} \\
  --batch-modify {template_file} \\
  --verbose \\
  --output modification_results.json
"""

print("📝 Execution Command (remove --dry-run):")
print(execution_command)

# Simulate the results that would be returned
simulated_results = {
    "modifications_applied": 3,
    "successful_updates": 3,
    "failed_updates": 0,
    "details": [
        {"identifier": "sample-id-1", "field": "title", "status": "success"},
        {"identifier": "sample-id-2", "field": "description", "status": "success"},
        {"identifier": "sample-id-3", "field": "keywords", "status": "success"}
    ]
}

print("\n📊 Simulated Results:")
print(f"  ✅ Modifications applied: {simulated_results['modifications_applied']}")
print(f"  ✅ Successful updates: {simulated_results['successful_updates']}")
print(f"  ❌ Failed updates: {simulated_results['failed_updates']}")

print("\n🔍 Individual Results:")
for detail in simulated_results["details"]:
    print(f"  • {detail['identifier']} ({detail['field']}) → {detail['status']}")

print("\n✅ Batch modification completed successfully!")
print("📋 All metadata updates applied to test environment")

In [ ]:
# Demonstrate verification and quality checking
console.print(Panel.fit(
    "[bold purple]Results Verification[/bold purple]\n\n"
    "🔍 Verify modifications were applied\n"
    "📊 Quality check after changes\n"
    "📈 Before/after comparison",
    title="✅ Verification"
))

print("🔍 Verification Steps:")
print("\n1. 📋 Check modification logs:")
verification_command1 = f"""
o-nakala-curator \\
  --api-key {TEST_API_KEY} \\
  --validate-metadata \\
  --scope datasets
"""
print(verification_command1)

print("2. 📊 Generate updated quality report:")
verification_command2 = f"""
o-nakala-curator \\
  --api-key {TEST_API_KEY} \\
  --quality-report \\
  --output quality_report_after.json
"""
print(verification_command2)

print("3. 👤 Check user data status:")
verification_command3 = f"""
o-nakala-user-info \\
  --api-key {TEST_API_KEY} \\
  --datasets-only
"""
print(verification_command3)

print("\n📈 What to verify:")
print("  ✅ All targeted records were updated")
print("  📊 Metadata quality scores improved")
print("  🔍 No unintended side effects")
print("  📋 Audit trail is complete")
print("  🌍 Multilingual metadata preserved")

print("\n💡 Pro Tips:")
print("  • Save before/after quality reports for comparison")
print("  • Monitor modification logs for patterns")
print("  • Test with small batches first")
print("  • Keep templates for reproducible workflows")

# Clean up temporary files
import shutil
shutil.rmtree(temp_dir)
print("\n🧹 Temporary files cleaned up")
print("✅ Batch modification demonstration complete!")

## Part 5: Advanced Usage and Best Practices

Learn advanced techniques and best practices for production use.

In [ ]:
# Demonstrate user info and account management
console.print(Panel.fit(
    "[bold magenta]User Account Management[/bold magenta]\n\n"
    "👤 Account information and permissions\n"
    "📊 Usage statistics and quotas\n"
    "📚 Collection and dataset overview",
    title="🔐 Account Management"
))

user_info_command = f"""
o-nakala-user-info \\
  --api-key {TEST_API_KEY} \\
  --collections-only
"""

print("📝 User Info Command:")
print(user_info_command)

print("\n📊 Available Information:")
print("  👤 User profile and permissions")
print("  📚 Collection count and details")
print("  📁 Dataset inventory")
print("  📈 Usage statistics")
print("  🔒 Access permissions")

In [ ]:
# Best practices and production tips
console.print(Panel.fit(
    "[bold green]Production Best Practices[/bold green]\n\n"
    "🔒 Security: Use environment variables for API keys\n"
    "📋 Validation: Always use dry-run mode first\n"
    "🔄 Automation: Integrate with CI/CD pipelines\n"
    "📊 Monitoring: Regular quality assessments",
    title="🎯 Best Practices"
))

print("🔐 Security Best Practices:")
print("  • Store API keys in environment variables")
print("  • Use different keys for test/production")
print("  • Rotate API keys regularly")
print("  • Never commit credentials to version control")

print("\n📋 Workflow Best Practices:")
print("  • Always validate with --dry-run first")
print("  • Use CSV configurations for reproducibility")
print("  • Maintain consistent metadata schemas")
print("  • Regular quality assessments")

print("\n🔄 Automation Integration:")
print("  • CI/CD pipeline integration")
print("  • Scheduled quality reports")
print("  • Batch processing workflows")
print("  • Error handling and retries")

In [ ]:
# Troubleshooting and error handling examples
console.print(Panel.fit(
    "[bold yellow]Troubleshooting Guide[/bold yellow]\n\n"
    "Common issues and solutions\n"
    "Error handling strategies",
    title="🔧 Troubleshooting"
))

print("❓ Common Issues and Solutions:")
print("\n🔑 Authentication Errors:")
print("  Problem: 401 Unauthorized")
print("  Solution: Check API key and environment variables")

print("\n📁 File Not Found Errors:")
print("  Problem: CSV or file paths not found")
print("  Solution: Verify file paths and working directory")

print("\n📋 Validation Errors:")
print("  Problem: Metadata validation failures")
print("  Solution: Use --dry-run to identify issues first")

print("\n🌐 Network Issues:")
print("  Problem: Connection timeouts")
print("  Solution: Check internet connection and API status")

print("\n🔍 Debug Mode:")
print("  Add --debug flag to CLI commands for detailed logging")
print("  Check log files for detailed error information")
print("  Use --dry-run for safe testing")

## 🎉 Workshop Conclusion

**Congratulations!** You've completed the O-Nakala Core workshop.

### What You've Learned

✅ **Installation**: Set up O-Nakala Core from PyPI  
✅ **Upload Workflows**: Batch upload with metadata  
✅ **Collection Management**: Organize datasets thematically  
✅ **Quality Curation**: Analyze and improve metadata quality  
✅ **CLI Tools**: Command-line interface for automation  
✅ **Python API**: Programmatic access for integration  
✅ **Best Practices**: Security, validation, and production tips  

### Next Steps

1. **Try with your data**: Apply these workflows to your research projects
2. **Explore documentation**: Check the [user guides](../../docs/user-guides/) for detailed information
3. **Production setup**: Configure your production NAKALA environment
4. **Automation**: Integrate with your research workflows

### Resources

- 📖 [Complete Documentation](../../docs/)
- 🔧 [Workflow Examples](../workflow_documentation/)
- 📊 [Sample Datasets](../sample_dataset/)
- 🌐 [NAKALA Platform](https://nakala.fr)
- 🧪 [Test Environment](https://test.nakala.fr)

---

**Thank you for participating in the O-Nakala Core workshop!**

*Built for digital humanities researchers and academic data management.*